# negative-back — worked example 2: negative_back agrees with autograd

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `negative-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A manual backward rule should reproduce what PyTorch's autograd computes. For `y = (-x).sum()`, autograd populates `x.grad` with all `-1`s. Our `negative_back` fed a grad_out of all ones must produce the identical tensor.

## Worked solution

We define `negative_back` returning `-grad_out`. To validate it, we create a leaf tensor `x` with `requires_grad=True`, compute `y = (-x).sum()`, and call `y.backward()`. Because `d(sum(-x))/dx = -1` everywhere, autograd writes `-1` into `x.grad`. Independently, the upstream gradient flowing into the negate op from a `.sum()` is all ones, so `negative_back(ones, out, x)` should equal `x.grad`. We seed, build a `(4,)` vector, run both paths, and print whether they match elementwise. This shows the hand rule and autograd agree.

In [ ]:
import torch as t

t.manual_seed(1)

def negative_back(grad_out, out, x):
    return -grad_out

x = t.randn(4, requires_grad=True)
y = (-x).sum()
y.backward()

x_detached = x.detach()
out = -x_detached
grad_out = t.ones_like(x_detached)  # upstream grad from .sum()
manual = negative_back(grad_out, out, x_detached)
print('autograd grad:', x.grad)
print('manual grad:  ', manual)
print('agree:', t.equal(manual, x.grad))